# 04 — Evaluate & Inference

Loads the fine-tuned Kronos model and runs inference on new OHLCV data.
Includes regression tests and a simple price forecast visualisation.


In [ ]:
import sys
sys.path.insert(0, '..')  # repo root

SYMBOL      = 'bnbusdt'
TIMEFRAME   = '1m'
GCS_BUCKET  = 'epochquant-training'
GCS_PROJECT = None

SAVE_PATH      = f'../output_models/{SYMBOL}_{TIMEFRAME}'
TOKENIZER_CKPT = f'{SAVE_PATH}/tokenizer_finetuned/checkpoints/best_model'
PREDICTOR_CKPT = f'{SAVE_PATH}/predictor_finetuned/checkpoints/best_model'
DATA_PATH      = f'gs://{GCS_BUCKET}/processed/{SYMBOL}_{TIMEFRAME}.csv'


In [ ]:
# ── Load fine-tuned models ────────────────────────────────────────
import torch
from model.kronos import Kronos, KronosTokenizer, KronosPredictor

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = KronosTokenizer.from_pretrained(TOKENIZER_CKPT)
model     = Kronos.from_pretrained(PREDICTOR_CKPT)

predictor = KronosPredictor(model, tokenizer, device=str(device))
print(f'Models loaded on {device}')


In [ ]:
# ── Load recent data from GCS ─────────────────────────────────────
from data.data_loader import load_from_gcs
import pandas as pd

df = load_from_gcs(DATA_PATH, gcs_project=GCS_PROJECT)
print(f'Loaded {len(df):,} rows')

# Use last 512 candles as lookback, predict next 60
LOOKBACK = 512
PRED_LEN = 60

df_input = df.tail(LOOKBACK).reset_index(drop=True)
x_ts = df_input['timestamps']

# Future timestamps
last_ts   = x_ts.iloc[-1]
freq      = pd.Timedelta(minutes=1)  # adjust for timeframe
y_ts      = pd.date_range(start=last_ts + freq, periods=PRED_LEN, freq=freq)

print(f'Input  : {x_ts.iloc[0]} → {x_ts.iloc[-1]}')
print(f'Predict: {y_ts[0]} → {y_ts[-1]}')


In [ ]:
# ── Run inference ─────────────────────────────────────────────────
pred_df = predictor.predict(
    df=df_input,
    x_timestamp=x_ts,
    y_timestamp=y_ts,
    pred_len=PRED_LEN,
    T=1.0, top_k=0, top_p=0.9,
    sample_count=5,
    verbose=True
)
print('Predictions shape:', pred_df.shape)
pred_df.head()


In [ ]:
# ── Plot: actual vs predicted close ──────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_input['timestamps'], df_input['close'], label='Historical', color='steelblue')
ax.plot(pred_df.index, pred_df['close'], label='Forecast', color='darkorange', linestyle='--')
ax.axvline(x_ts.iloc[-1], color='gray', linestyle=':', alpha=0.7, label='Forecast start')
ax.set_title(f'Kronos Forecast — {SYMBOL.upper()} {TIMEFRAME}')
ax.set_xlabel('Time')
ax.set_ylabel('Price')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── Run regression tests ──────────────────────────────────────────
import subprocess
result = subprocess.run(['python', '-m', 'pytest', 'tests/', '-v'], cwd='..')
print('Tests exit code:', result.returncode)
